In [ ]:
# %pip install langchain openai langchain-core langchain-openai langchain-community

In [ ]:
import os
from dotenv import load_dotenv

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
property_info = """
    123 Main Street, Austin TX

    25,000 SF Retail Center
    95% Occupied
    Anchored by Starbucks
    Average Household Income: $145,000
    Traffic Count: 42,000 VPD
    Price: $8.5M
"""

In [ ]:
from langchain_openai import ChatOpenAI
# from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

# Generate property descriptioin using LLM and prompt template
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI


In [ ]:

llm = ChatOpenAI(model="gpt-4o-mini")

listing_prompt = PromptTemplate(
    input_variables=["property"],
    template="""
        You are a commercial real estate broker.

        Create a professional CRE listing description.

        Property Information:
        {property}
    """
)

chain = listing_prompt | llm

result = chain.invoke({
    "property": property_info
})

print(result.content)

In [ ]:
#Let's create a LinkedIn post for the above listing using the generated description
linkedin_prompt = PromptTemplate(
    input_variables=["listing"],
    template="""
        Create a professional LinkedIn post for commercial real estate investors.

        Listing Description:
        {listing}

        Include:
        - Hook
        - Key highlights
        - Call to action
"""
)

linkedin_chain = linkedin_prompt | llm

linkedin_post = linkedin_chain.invoke({
    "listing": result.content
})

print(linkedin_post.content)

In [ ]:
# Generate a commercial real estate outreach email for the above listing using the generated description
email_prompt = PromptTemplate(
    input_variables=["listing"],
    template="""
        Write a commercial real estate outreach email.

        Property Details:
        {listing}

        Requirements:
        - Subject line
        - Short email
        - Strong call to action
"""
)

email_chain = email_prompt | llm

email = email_chain.invoke({
    "listing": result.content
})

print(email.content)

In [ ]:
# Parse Output into JSON
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

class PropertySummary(BaseModel):
    property_type: str
    city: str
    price: float
    key_tenants: list[str]

parser = PydanticOutputParser(
    pydantic_object=PropertySummary
)

prompt = PromptTemplate(
    template="""
        Extract information from the property.

        {format_instructions}

        Property:
        {property}
    """,
    input_variables=["property"],
    partial_variables={
        "format_instructions":
        parser.get_format_instructions()
    }
)

chain = prompt | llm | parser

result = chain.invoke({
    "property": property_info
})

print(result)
